# Non-Linear Classification and Production ML Systems

## 1. Introduction to Supervised Learning
In Supervised Learning, we aim to learn a mapping function $f: X \to Y$ that relates input features ($X$) to discrete labels ($Y$). Unlike regression, which predicts continuous values, **Classification** deals with predicting categories.

**The Problem Setup:**
We are given a training set D $\mathcal{D} = \{(x^{(1)}, y^{(1)})\}, ..., \{(x^{(m)}, y^{(m)})\}$. The goal is to build a model that generalizes to unseen data, maximizing the probability that our predicted label $\hat{y}$ matches the true label $y$.

**Real-World Context:**
Consider a bank's automated credit system. Features ($X$) include income, credit history, and age. The label ($Y$) is binary: $\{0: \text{Denied}, 1: \text{Approved}\}$.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Mock setup for demonstration
# In a real scenario, you would load: df = pd.read_csv('loan_data.csv')
data = {'Income': [50, 80, 20, 30, 70], 'Age': [25, 45, 18, 22, 35], 'Target': ['Approved', 'Approved', 'Denied', 'Denied', 'Approved']}
df = pd.DataFrame(data)

# Encoding categorical target labels
le = LabelEncoder()
y = le.fit_transform(df['Target'])
X = df.drop('Target', axis=1)

# Splitting into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data Split Complete.")

## 2. Decision Trees (DT)

### 2.1 Intuition: Recursive Partitioning
A Decision Tree builds a flowchart-like structure to classify data. It works by **recursive partitioning**—splitting the data into subsets based on feature values (e.g., "Is Income > $50k?").

### 2.2 Mathematics of Purity
* **Gini Impurity:** $Gini = 1 - \sum_{i=1}^{C} (p_i)^2$
* **Entropy:** $H = -\sum_{i=1}^{C} p_i \log_2(p_i)$

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize with pruning parameters to prevent overfitting
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=5, min_samples_split=10)
dt_model.fit(X_train, y_train)

print(f"Decision Tree Trained. Accuracy: {dt_model.score(X_test, y_test)}")

## 3. K-Nearest Neighbors (KNN)

KNN identifies the $K$ closest training points in the feature space and assigns the most common label among them.

**Distance Metric (Euclidean):** $d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$

*Note: KNN is distance-based, so feature scaling is mandatory.*

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Scaling is crucial for KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train)

print(f"KNN Trained. Accuracy: {knn.score(X_test_scaled, y_test)}")

## 4. Support Vector Machines (SVM)

SVM finds the hyperplane that maximizes the **Margin**—the distance between the boundary and the closest data points (**Support Vectors**). For non-linear data, we use the **Kernel Trick**.

In [ ]:
from sklearn.svm import SVC

# Using RBF kernel for non-linear decision boundaries
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_model.fit(X_train_scaled, y_train)

print(f"SVM Trained. Accuracy: {svm_model.score(X_test_scaled, y_test)}")

## 5. Azure Machine Learning (Azure ML)

In enterprise environments, we move from local scripts to **Azure ML** to manage the lifecycle.

### Azure ML Workspace
The hub for all ML activities. It manages:
* **Assets:** Data, Models, and Docker Environments.
* **Compute:** Managed GPU/CPU Clusters.

### Azure ML Pipelines
A pipeline is a Directed Acyclic Graph (DAG) of modular steps:
* **Data Prep:** Cleaning data at scale.
* **Sweep Jobs:** Parallel hyperparameter tuning for $C$ and $\gamma$.
* **Registration:** Versioning the validated model.

### CI/CD
Continuous Integration (CI) automates training upon code changes. Continuous Deployment (CD) serves the model via **Online Endpoints**.

In [ ]:
# Note: This requires azure-ai-ml package installed
# !pip install azure-ai-ml azure-identity

from azure.ai.ml import MLClient, command
from azure.identity import DefaultAzureCredential

# Placeholder for Workspace connection
try:
    ml_client = MLClient(DefaultAzureCredential(), "subscription_id", "resource_group", "workspace_name")

    # Define a simple training command job
    job = command(
        code="./src", # Folder containing training script
        command="python train.py --data ${{inputs.data}}",
        inputs={"data": "loan_dataset:1"},
        environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu:1",
        compute="cpu-cluster"
    )
    print("Azure Job defined successfully.")
except Exception as e:
    print("Azure connection requires local credentials. Job definition ready for cloud deployment.")

## 6. Model Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = svm_model.predict(X_test_scaled)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 📝 End-of-Topic Questions

### Conceptual Questions:
1. **Greedy Splitting:** Why is a Decision Tree considered "greedy"?
2. **Resource Management:** Why is an Azure "Sweep Job" more efficient than a local `for` loop?
3. **Endpoint Versioning:** How does Azure's Online Endpoint support Blue/Green deployment?

### Thought Experiment:
A feature is "missing" due to a DB error. How does **Azure Model Monitoring** detect this as "Data Drift"?